In [ ]:
import json
from langchain_openai import OpenAI
from langchain_community.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.tools import Tool
import re
import os
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.chains import RetrievalQA
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from dotenv import load_dotenv

# 환경변수 설정
load_dotenv()

True

In [ ]:
def get_llm(model = 'gpt-4o-mini'):
    llm = ChatOpenAI(model = model)
    return llm

# 01. Vector_DB & Retriever Tool

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.tools.retriever import create_retriever_tool

# PDF Vector DB 구성

# Chroma DB 불러오기
pdf_database = Chroma(
    collection_name="rag_multi_modal",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_derma_studies"
)

# Retriever 생성
pdf_retriever = pdf_database.as_retriever()


# # Retriever Tool 설정
# pdf_retriever_tool = create_retriever_tool(
#     pdf_retriever,
#     name="derma_search",
#     description="Searches any questions related to dermatological topics. Always use this tool when user query is related to dermatology!",
# )

In [ ]:
# Product Vector Db 구성

prod_database = Chroma(
    collection_name='product_list',
    embedding_function=OpenAIEmbeddings(),
    persist_directory='./chroma_product_list_v1_0831'
)

# retriever 생성
prod_retriever = prod_database.as_retriever()


# # Retriever Tool 설정
# product_retriever_tool = create_retriever_tool(
#     product_retriever,
#     name = "product_search",
#     description="Searches any questions related to skincare products. Use this tool when the user's query is related to skincare products, such as recommendations or information.",
# )

# 02. Tool 지정

https://python.langchain.com/v0.1/docs/modules/tools/custom_tools/

### (1) 키워드 추출 및 retriever 정의

In [ ]:
# LLM을 이용한 키워드 추출 함수
def extract_keywords_from_text(query):
    # LLM에게 키워드를 추출하도록 프롬프트를 생성
    prompt_template = "다음 문장에서 핵심 키워드를 추출해줘: '{query}'"
    prompt = PromptTemplate(input_variables=["query"], template=prompt_template)

    llm = OpenAI()
    # LLM에 프롬프트를 전달하여 키워드 추출
    chain = prompt | llm | StrOutputParser()

    response = chain.invoke(query)
    # 키워드를 쉼표로 구분한 결과를 리스트로 변환
    keywords = [kw.strip() for kw in response.split(",")]
    return keywords

# Chroma DB에서 검색하는 함수
def search_chroma_db(keywords, retriever):
    # 키워드들을 조합하여 검색 수행
    query = " ".join(keywords)  # 키워드들을 공백으로 구분하여 하나의 쿼리로 결합
    results = retriever.invoke(query)
    return results

In [ ]:
query = '모공관리 방법을 알려줘.'

keywords = extract_keywords_from_text(query)

documents = search_chroma_db(keywords, pdf_retriever)

Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')
Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')


Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')


In [ ]:
documents

[Document(metadata={'doc_id': '2319cfba-4e59-4a78-8c38-320cbb0cdacf'}, page_content='Screenshot of a Korean government health-related website with a blue and white interface, featuring a laptop displaying a bar chart, medical items, various navigational buttons, and sections for application, information, reporting, and usage guides.'),
 Document(metadata={'doc_id': 'fe39efbf-1154-449d-971f-67a094357295'}, page_content='Bar graph with 95% confidence intervals in Korean, comparing three categories: 주근깨 (freckles), 검버섯 (liver spots), and 기미 (melasma). Y-axis labeled 개선정도 (improvement level).'),
 Document(metadata={'doc_id': '78388e39-f19a-4536-9c24-bbfb7b2d91c9'}, page_content='Bar graph with error bars showing 95% confidence intervals for three groups labeled in Korean, with a y-axis labeled "개선정도" ranging from 1.5 to 3.5.'),
 Document(metadata={'doc_id': '48fe2a55-6a1f-4090-bee9-007fec87810a'}, page_content='Bar graph with error bars showing 95% confidence intervals for three groups lab

### (2) Tool 정의

In [ ]:
from langchain.agents import tool

@tool
def get_skincare_advices(query: str) -> str:
    """
    Provides expert advice and tips related to dermatological topics, shuch as skincare routines, and skin care in general.
    Use this tool specifically when the user's query is about skincare methods, routines, or general advice for maintaining healthy skin.

    Args:
        query (str): The user's query asking for skincare advice, tips, or information about skincare routines.

    Returns:
        str: A detailed, formatted response based on the embedded knowledge, including step-by-step skincare routines, product application tips, and personalized advice according to the user's skin type or concerns.
    """

    # 응답을 생성하는 함수
    def generate_response(query, documents):

        # Document 객체에서 텍스트를 추출하여 리스트로 변환
        document_texts = [doc.page_content for doc in documents]  # 각 Document 객체의 텍스트 추출

        # 프롬프트 템플릿 설정

        template = """
            You are an expert dermatologist tasked with providing personalized skincare advice based on the information provided.
            Analyze the following content carefully, and generate a detailed response to the user's query in Korean.

            ### User's Query:
            {query}

            ### Relevant Information:
            {context}

            ### Your Advice:
            Provide specific skincare advice tailored to the user's concerns, including steps, product recommendations, and any other relevant tips. Answer concisely and professionally.
            """

        context = {"context": " ".join(document_texts)}

        prompt = PromptTemplate.from_template(template)

        llm =  ChatOpenAI(model_name = 'gpt-4o-mini')

        output_parser = StrOutputParser()


        chain = prompt | llm | output_parser
        responses = chain.invoke({'context': context, 'query':query})

        return responses

    # 쿼리 처리 및 제품 추천
    keywords = extract_keywords_from_text(query)

    documents = search_chroma_db(keywords, pdf_retriever)

    skincare_advices = generate_response(query, documents)

    return skincare_advices

In [ ]:
@tool
def get_prod_recommendation(query: str) -> list:
    """
    Searches any questions related to skincare products recommendations.
    Use this tool when the user's query is related to skincare products recommendations.

    Args:
        query (str): The user's query describing their skin concerns or the type of skincare product they are looking for.

    Returns:
        list: A list of recommended skincare products formatted according to the prompt defined in the function below, including product names, brand, price, url, and summary for the user's skin concern.
    """

    # 응답을 생성하는 함수
    def generate_response(documents):
    #답변을 저장할 빈 리스트
        responses = []

        for idx, doc in enumerate(documents[:3], start=1):  # 상위 3개의 문서만 사용
            # 딕셔너리로 각 제품의 정보를 저장
            context = {
                "name": doc.metadata['name'],  # 인덱스 번호를 제품명 앞에 추가
                "brand": doc.metadata['brand'],
                "price": doc.metadata['price'],
                "url": doc.metadata['url'],
                "Summary": doc.metadata['Summary']
            }

            url = doc.metadata['url']
            # idx = idx
            # name = context['제품명']
            # brand = context['브랜드']
            # price = context['가격']
            # url = context['구매링크']
            # Summary = context['리뷰요약']

            # 프롬프트 템플릿 설정
            template = """
            you are a dermatologist working in dermatoly field about 10 years. Please generate an answer in the following [FORMAT].

            You are provided with dictionary type data which is the information of a dermal cosmetic product:
            {context}

            FORMAT:
            - 제품명: "name",
            - 브랜드: "brand",
            - 가격: 'price',
            - 링크 :'[구매링크]({url})',
            - 리뷰요약: 'Summary'

            please do this performing as post processing.
            1 - Please generate answer after summarize the 'Summary' in 3 sentences.
            2 - write the '원' at the end of the price(int).
            3 - Please Delete punctuations that you think useless in the message.
            """

            prompt = PromptTemplate.from_template(template)

            model =  ChatOpenAI(model_name = 'gpt-4o')

            output_parser = StrOutputParser()


            chain = prompt | model | output_parser
            response = chain.invoke({'context': context, 'url':url})

            responses.append(response)

        return responses

    # 쿼리 처리 및 제품 추천
    keywords = extract_keywords_from_text(query)

    documents = search_chroma_db(keywords, prod_retriever)

    prod_recommendation = generate_response(documents)

    return prod_recommendation

# 03. Agent 생성

In [ ]:
from langchain_community.tools.convert_to_openai import format_tool_to_openai_function
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.format_scratchpad import format_to_openai_function_messages
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain.agents import AgentExecutor


# langchain agent tools 설정
tools = [get_skincare_advices, get_prod_recommendation]

# llm 모델 정의
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# llm 모델에 tools 바인딩
llm_with_tools = llm.bind(
    functions=[format_tool_to_openai_function(t) for t in tools])

# Prompt 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but don't know current events",
        ),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)


# agent 정의
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_function_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | OpenAIFunctionsAgentOutputParser()
)

# agent_executor 정의
agent_executor = AgentExecutor(
    agent=agent, tools=tools, handle_parsing_errors=True, verbose=True
)

# 04.Agent 실행

In [ ]:
# query = "모공 관리 방법을 알려줄래?"
query = "모공에 좋은 제품을 추천해줄래?"
result = agent_executor.invoke({"input": query})



> Entering new AgentExecutor chain...


Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')



Invoking: `get_prod_recommendation` with `{'query': '모공'}`




Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')
Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')
Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')
Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')
Failed to batch 

['- 제품명: "메디큐브제로모공원데이세럼"\n- 브랜드: "메디큐브"\n- 가격: \'36400원\'\n- 링크: \'[구매링크](https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000206184&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%BC%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_number=36)\'\n- 리뷰요약: \'사용 후 모공이 약간 조이는 느낌과 피부 결이 나아지는 것으로 느끼지만 드라마틱한 효과는 확인되지 않았습니다. 여름철 끈적임이 있을 수 있으며 다양한 피부 타입에 적합한 성분이 포함되어 있습니다. 꾸준한 사용이 필요하지만 전반적으로 긍정적인 반응입니다.\'', '- 제품명: "비긴스포어퍼펙팅세럼",\n- 브랜드: "비긴스",\n- 가격: 32000원,\n- 링크: \'[구매링크](https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000183156&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%B8%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_number=19)\',\n- 리뷰요약: 사용자들은 제

Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')


모공에 좋은 제품을 몇 가지 추천해드릴게요:

1. **메디큐브 제로 모공 원데이 세럼**
   - **브랜드**: 메디큐브
   - **가격**: 36,400원
   - **링크**: [구매링크](https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000206184&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%BC%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_number=36)
   - **리뷰 요약**: 사용 후 모공이 약간 조이는 느낌과 피부 결이 나아지는 것으로 느끼지만 드라마틱한 효과는 확인되지 않았습니다. 여름철 끈적임이 있을 수 있으며 다양한 피부 타입에 적합한 성분이 포함되어 있습니다. 꾸준한 사용이 필요하지만 전반적으로 긍정적인 반응입니다.

2. **비긴스 포어 퍼펙팅 세럼**
   - **브랜드**: 비긴스
   - **가격**: 32,000원
   - **링크**: [구매링크](https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000183156&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%B8%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%81%ED%92%88%E

Failed to batch ingest runs: LangSmithError('Failed to POST https://api.smith.langchain.com/runs/batch in LangSmith API. HTTPError(\'403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Forbidden"}\')')


In [ ]:
result['output']

'모공에 좋은 제품을 몇 가지 추천해드릴게요:\n\n1. **메디큐브 제로 모공 원데이 세럼**\n   - **브랜드**: 메디큐브\n   - **가격**: 36,400원\n   - **링크**: [구매링크](https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000206184&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%BC%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_number=36)\n   - **리뷰 요약**: 사용 후 모공이 약간 조이는 느낌과 피부 결이 나아지는 것으로 느끼지만 드라마틱한 효과는 확인되지 않았습니다. 여름철 끈적임이 있을 수 있으며 다양한 피부 타입에 적합한 성분이 포함되어 있습니다. 꾸준한 사용이 필요하지만 전반적으로 긍정적인 반응입니다.\n\n2. **비긴스 포어 퍼펙팅 세럼**\n   - **브랜드**: 비긴스\n   - **가격**: 32,000원\n   - **링크**: [구매링크](https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000183156&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%B8%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%8